In [ ]:
from sentence_transformers import CrossEncoder, CrossEncoderTrainer, SentenceTransformer
from sentence_transformers.evaluation.SequentialEvaluator import SequentialEvaluator
from sentence_transformers.util import mine_hard_negatives
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
from sentence_transformers.cross_encoder import CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.evaluation import (
    CrossEncoderClassificationEvaluator,
    CrossEncoderRerankingEvaluator,
)
from typing import Optional,List,Dict,Any
from datasets import Dataset
import pandas as pd
from pathlib import Path
from huggingface_hub import create_repo, upload_folder
import sys, os
sys.path.append(os.path.abspath("../../../"))
from backend.database.config.config import settings

c:\Users\johnk\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
p_folder = f'{os.getcwd()}'.split('\\')[:-1]
path_folder = '//'.join(p for p in p_folder)+'//'
print(path_folder)

c://Users//johnk//Documents//GitHub//AILA-application//backend//evaluation//


In [ ]:

class CrossEncoderFinetuning:
    def __init__(self, path: str, model_id: str = 'BAAI/bge-reranker-v2-m3'):
        self.model_id = model_id
        self.model = CrossEncoder(self.model_id, num_labels=1, max_length=512)
        self.path = path  # base path to your data folder

    def data_preparation(self, df_feedback: Optional[pd.DataFrame] = None) -> Dataset:
        # Load base data
        base_csv = Path(self.path) / 'synthetic_data_new' / 'new_queries.csv'
        df_base = pd.read_csv(base_csv)

        queries, labels, answers = [], [], []

        # Expect df_base has columns: anchor, positive
        for i in range(len(df_base)):
            query = df_base.at[i, 'anchor']
            pos = df_base.at[i, 'positive']
            queries.append(query)
            answers.append(pos)
            labels.append(1)

        # Optional human feedback: columns query, correct_answer, negative_answer
        if df_feedback is not None and len(df_feedback) > 0:
            for i in range(len(df_feedback)):
                q = df_feedback.at[i, 'query']
                ca = df_feedback.at[i, 'context']
                na = df_feedback.at[i, 'negative_answer']
                # replicate the query once per answer to keep lists aligned
                queries.extend([q, q])
                answers.extend([ca, na])
                labels.extend([1, 0])

        dataset = Dataset.from_dict({'query': queries, 'response': answers, 'label': labels})
        return dataset

    def push_to_hub(
        self,
        model: CrossEncoder,
        trainer: CrossEncoderTrainer,
        model_name: str,
        namespace: str = 'IoannisKat1'
    ) -> None:
        best_ckpt = trainer.state.best_model_checkpoint
        out_dir = model_name
        if best_ckpt is None:
            save_dir = Path(out_dir) / 'final'
            save_dir.mkdir(parents=True, exist_ok=True)
            model.save(str(save_dir))
        else:
            save_dir = Path(best_ckpt) / 'export'
            save_dir.mkdir(parents=True, exist_ok=True)
            model.save(str(save_dir))

        repo_id = f"{namespace}/{model_name}_ft"
        create_repo(repo_id, private=False, exist_ok=True, token=settings.HF_TOKEN)
        upload_folder(
            repo_id=repo_id,
            folder_path=str(save_dir),
            commit_message='Finetuned Reranker',
            token=settings.HF_TOKEN
        )
        print(f"Pushed {save_dir} -> {repo_id}")

    def cross_encoder_tuning(
        self,
        df_feedback: Optional[pd.DataFrame] = None,
        epochs: int = 20,
        batch_size: int = 16,
        lr: float = 2e-5,
        warmup_ratio: float = 0.1,
        eval_split: float = 0.2,
        output_path: str = 'bge-reranker-ft'
    ) -> None:
        dataset = self.data_preparation(df_feedback)
        if len(dataset) == 0:
            raise ValueError("No data was provided for cross encoder finetuning")

        dataset = dataset.shuffle(seed=43).train_test_split(test_size=eval_split)
        train_dataset = dataset['train']
        eval_dataset = dataset['test']

        def to_list(dataset, key):
            col = dataset[key]
            if isinstance(col, list):
                return col
            to_pylist = getattr(col, "to_pylist", None)
            return to_pylist() if callable(to_pylist) else list(col)

        train_responses = to_list(train_dataset, "response")
        eval_responses  = to_list(eval_dataset,  "response")
        N_docs = len(dataset)
        # Combine and deduplicate while keeping order
        seen = set()
        corpus = []
        for resp in train_responses + eval_responses:
            if resp not in seen:
                seen.add(resp)
                corpus.append(resp)

        # Build a corpus for negative mining from ALL responses
        # corpus = list(set(train_dataset['response'] + eval_dataset['response']))
        corpus_size = len(corpus)
        if corpus_size < 2:
            raise ValueError("Not enough unique responses to mine negatives.")

        # Keep negatives sane relative to corpus size
        train_num_negs = min(5, max(1, corpus_size - 1))

        print(train_num_negs)
        print(min(100, max(5, N_docs-1)) )

        embedding_model = SentenceTransformer(
            "sentence-transformers/static-retrieval-mrl-en-v1", device="cpu"
        )

        hard_train_dataset = mine_hard_negatives(
            train_dataset,
            embedding_model,
            corpus=corpus,
            num_negatives=train_num_negs,     # negatives per (query, positive)
            margin=0,
            range_min=0,
            range_max=min(100, max(5, N_docs-1)) ,
            sampling_strategy="top",
            batch_size=4096,
            output_format="labeled-pair",   # (query, passage, label)
            use_faiss=False,
        )

        loss = BinaryCrossEntropyLoss(self.model)

        # --- Evaluators ---
        # Classification evaluator
        pairs = list(zip(eval_dataset["query"], eval_dataset["response"]))
        labels = eval_dataset["label"]
        dev_evaluator = CrossEncoderClassificationEvaluator(
            sentence_pairs=pairs,
            labels=labels,
            name="cls-dev",
        )

        # Reranking evaluator built from mined hard negatives on eval split
        eval_num_negs = min(5, max(1, corpus_size - 1))
        print(eval_num_negs)
        hard_eval_dataset = mine_hard_negatives(
            eval_dataset,
            embedding_model,
            corpus=corpus,
            num_negatives=eval_num_negs,
            batch_size=4096,
            include_positives=True,
            output_format="n-tuple",  # returns {"query","response","neg_0","neg_1",...}
            use_faiss=False,
        )

        # Convert n-tuple rows into the structure expected by CERerankingEvaluator
        rr_samples = []
        for sample in hard_eval_dataset:
            q = sample["query"]
            pos = sample["response"]
            # Collect all negative columns dynamically
            negatives = [v for k, v in sample.items() if k.startswith("neg_")]
            docs = [pos] + negatives  # positive included
            rr_samples.append({"query": q, "positive": [pos], "documents": docs})

        reranking_evaluator = CrossEncoderRerankingEvaluator(
            samples=[
                {
                    "query": sample["query"],
                    "positive": [sample["response"]],
                    "documents": [sample[column_name] for column_name in hard_eval_dataset.column_names[2:]],
                }
                for sample in hard_eval_dataset
            ],
            batch_size=batch_size,
            name="gooaq-dev",
            always_rerank_positives=False,
        )

        # Optional: quick pre-train reranking score (kept for visibility)
        _ = reranking_evaluator(self.model)

        evaluator = SequentialEvaluator([reranking_evaluator, dev_evaluator])

        # If you're on CPU, keep bf16/amp off
        args = CrossEncoderTrainingArguments(
            output_dir=output_path,
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=lr,
            warmup_ratio=warmup_ratio,
            fp16=False,
            bf16=False,
            dataloader_num_workers=4,
            load_best_model_at_end=True,
            # Metric name must match the evaluator’s reported keys.
            # CERerankingEvaluator typically reports "ndcg@10" → becomes "eval_rr-dev_ndcg@10"
            metric_for_best_model="eval_rr-dev_ndcg@10",
            greater_is_better=True,
            eval_strategy="steps",
            eval_steps=4000,
            save_strategy="steps",
            save_steps=4000,
            save_total_limit=2,
            logging_steps=20,
            logging_first_step=True,
            seed=12,
        )

        trainer = CrossEncoderTrainer(
            model=self.model,
            args=args,
            train_dataset=hard_train_dataset,
            loss=loss,
            evaluator=evaluator,
        )
        trainer.train()

        # Final evaluation
        evaluator(self.model)

        # Push to Hub
        self.push_to_hub(self.model, trainer, 'bge_reranker')

In [ ]:
df = pd.read_csv(f'{path_folder}synthetic_dataset/feedback.csv')
cross_encoder_funetuning = CrossEncoderFinetuning(path=path_folder)
# cross_encoder_funetuning.cross_encoder_tuning()
cross_encoder_funetuning.cross_encoder_tuning(df_feedback=df)